In [1]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import subprocess
subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "transformers>=4.41.0", "peft", "accelerate", "datasets", "scikit-learn"
], check=True)

import transformers
print("Transformers:", transformers.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 42.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


Transformers: 5.3.0


In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os, json
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


Torch: 2.9.0+cu126
GPU: Tesla T4


In [3]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
MODEL_NAME = "google/electra-base-discriminator"
DATA_DIR   = "/kaggle/input/datasets/karthiksekar1821/humaid"
OUTPUT_DIR = "/kaggle/working/electra_humaid"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
# ── Cell 4: Load Dataset ──────────────────────────────────────────────────────
dataset = load_dataset(
    "parquet",
    data_files={
        "train":      f"{DATA_DIR}/train.parquet",
        "validation": f"{DATA_DIR}/val.parquet",
        "test":       f"{DATA_DIR}/test.parquet",
    }
)
print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 53531
    })
    validation: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 7793
    })
    test: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 15160
    })
})


In [5]:
# ── Cell 5: Label Mapping ─────────────────────────────────────────────────────
with open(f"{DATA_DIR}/label_mapping.json") as f:
    mapping = json.load(f)

label2id   = mapping["label2id"]
id2label   = {int(k): v for k, v in mapping["id2label"].items()}
num_labels = len(label2id)
print(f"{num_labels} labels:", list(label2id.keys()))


10 labels: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support']


In [6]:
# ── Cell 6: Tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["tweet_text"], truncation=True, max_length=MAX_LEN)

dataset = dataset.map(tokenize, batched=True, remove_columns=["tweet_text"])
dataset = dataset.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in dataset["train"].column_names:
    keep_cols.append("token_type_ids")
dataset.set_format("torch", columns=keep_cols)
print("Tokenization complete. Columns:", keep_cols)


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/53531 [00:00<?, ? examples/s]

Map:   0%|          | 0/7793 [00:00<?, ? examples/s]

Map:   0%|          | 0/15160 [00:00<?, ? examples/s]

Tokenization complete. Columns: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']


In [7]:
# ── Cell 7: Model ─────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# ELECTRA has a multi-layer classifier head — check the final projection layer
classifier_out = model.classifier.out_proj.out_features
print(f"Classifier output size: {classifier_out} (should be {num_labels})")
assert classifier_out == num_labels, "Classifier size mismatch!"


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Classifier output size: 10 (should be 10)


In [8]:
# ── Cell 8: Metrics ───────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


In [9]:
# ── Cell 9: Training Args ─────────────────────────────────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [10]:
# ── Cell 10: Train ────────────────────────────────────────────────────────────
print("\n--- Starting Training ---")
trainer.train()



--- Starting Training ---


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Macro F1
1,1.913373,1.406657,0.719980
2,1.194139,1.292194,0.747609
3,0.980957,1.297616,0.750742
4,0.806712,1.375849,0.752004
5,0.674875,1.463968,0.750390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=8365, training_loss=1.1140110013588986, metrics={'train_runtime': 2644.8082, 'train_samples_per_second': 101.2, 'train_steps_per_second': 3.163, 'total_flos': 9057186392509836.0, 'train_loss': 1.1140110013588986, 'epoch': 5.0})

In [11]:
print("\n--- Evaluating on Test Set ---")
predictions = trainer.predict(dataset["test"])
preds       = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids
test_results = predictions.metrics
print("Test metrics:", test_results)
print("\nClassification Report:")
print(classification_report(true_labels, preds, target_names=list(id2label.values())))


--- Evaluating on Test Set ---


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test metrics: {'test_loss': 1.3446518182754517, 'test_macro_f1': 0.7605784641851047, 'test_runtime': 41.1276, 'test_samples_per_second': 368.609, 'test_steps_per_second': 11.525}

Classification Report:
                                        precision    recall  f1-score   support

                    caution_and_advice       0.72      0.71      0.71      1070
      displaced_people_and_evacuations       0.88      0.91      0.89       790
     infrastructure_and_utility_damage       0.81      0.83      0.82      1617
                injured_or_dead_people       0.91      0.94      0.92      1447
               missing_or_found_people       0.76      0.76      0.76        72
                      not_humanitarian       0.62      0.58      0.60      1245
            other_relevant_information       0.62      0.51      0.56      2407
              requests_or_urgent_needs       0.63      0.58      0.61       521
rescue_volunteering_or_donation_effort       0.84      0.92      0.88      4

In [12]:
# ── Cell 12: Save ─────────────────────────────────────────────────────────────
import shutil

BEST_MODEL_DIR = f"{OUTPUT_DIR}/best_model"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Save model and tokenizer
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Manually patch config to ensure num_labels is saved correctly
config_path = f"{BEST_MODEL_DIR}/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

config["num_labels"] = num_labels
config["id2label"]   = {str(k): v for k, v in id2label.items()}
config["label2id"]   = label2id

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("✅ Config patched with num_labels, id2label, label2id")

# Save results
report = classification_report(
    true_labels, preds,
    target_names=list(id2label.values()),
    output_dict=True,
)
with open(f"{OUTPUT_DIR}/test_results.json", "w") as f:
    json.dump({"overall": test_results, "per_class": report}, f, indent=4)

# Zip for easy download
shutil.make_archive("/kaggle/working/electra_best_model", "zip", OUTPUT_DIR, "best_model")
print(f"✅ Zipped: /kaggle/working/electra_best_model.zip")

# Verify config saved correctly
with open(config_path) as f:
    saved = json.load(f)
print(f"\nVerification:")
print(f"  num_labels : {saved['num_labels']}")
print(f"  id2label[0]: {saved['id2label']['0']}")
print(f"  id2label[9]: {saved['id2label']['9']}")

print(f"\n✅ Done. Download electra_best_model.zip from the Output tab.")
print("   Then go to Output tab → New Dataset to persist it permanently.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Config patched with num_labels, id2label, label2id
✅ Zipped: /kaggle/working/electra_best_model.zip

Verification:
  num_labels : 10
  id2label[0]: caution_and_advice
  id2label[9]: sympathy_and_support

✅ Done. Download electra_best_model.zip from the Output tab.
   Then go to Output tab → New Dataset to persist it permanently.
